# PART1
Fetching and storing the data from NCBI, Uniprot and AFDB

In [ ]:
import os
import pandas as pd
import requests
import shutil
from io import StringIO
from Bio.SeqIO import parse
from tqdm import tqdm

In [ ]:
# Definitions
proteome_entries = {
    "K12": "UP000000625",
    "Sakai": "UP000000558",
    "EAEC": "UP000000746",
    "APEC": "UP000008216",
    "ETEC": "UP000001122",
    "EPEC": "UP000008205",
    "AIEC": "UP000008614",
    "ExPEC": "UP000000747",
    "UPEC": "UP000001410",
    # "HS": "UP000001123",
}
data_dir = os.path.join("..", "data")
preprocess_data_dir = os.path.join(data_dir, "preprocess")
if not os.path.exists(preprocess_data_dir):
    os.makedirs(preprocess_data_dir)

In [ ]:
# Uniprot proteome query URL
Uniprot_proteome_query_head = "https://rest.uniprot.org/proteomes/stream?fields=upid%2Corganism%2Corganism_id%2Cgenome_assembly&format=tsv&query="

In [ ]:
# Fetch data from Uniprot
Uniprot_proteome_query_URL = Uniprot_proteome_query_head
for name, UP in proteome_entries.items():
    Uniprot_proteome_query_URL += "upid%3A" + UP + "+OR+"
Uniprot_proteome_query_URL = Uniprot_proteome_query_URL[:-4]
Uniprot_response = requests.get(Uniprot_proteome_query_URL)
if Uniprot_response.status_code == 200:
    proteome_df = pd.read_csv(StringIO(Uniprot_response.text), sep="\t")
    print(f"Success: {len(proteome_df)} proteome entries fetched")
    proteome_df["Genome assembly ID"] = proteome_df["Genome assembly ID"].str.replace("GCA_", "GCF_")
else:
    proteome_df = pd.DataFrame()
    print(f"Error: {Uniprot_response.status_code}")
proteome_df.head()

In [ ]:
# NCBI genome assembly URL
NCBI_URL_template = "https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/{}/download?include_annotation_type=GENOME_GBFF"

In [ ]:
# Fetch and Extract gbff from genome assembly data
NCBI_dir = os.path.join(data_dir, "NCBI")
if not os.path.exists(NCBI_dir):
    os.makedirs(NCBI_dir)
for GCF_id in proteome_df["Genome assembly ID"]:
    err = 1
    GCF_prefix, version = GCF_id.split(".")
    version = int(version)
    while err:
        GCF_id_correct = GCF_prefix + "." + str(version)
        if not os.path.exists(os.path.join(NCBI_dir, GCF_id_correct + ".gbff")):
            if version < 1:
                print(f"Error: {GCF_prefix} genome assembly data not found")
                break
            zip_path = os.path.join(NCBI_dir, GCF_id_correct + ".zip")
            version = int(version)
            if not os.path.exists(zip_path):
                NCBI_response = requests.get(NCBI_URL_template.format(GCF_id_correct))
                if NCBI_response.status_code == 200:
                    print(f"Success: {GCF_id_correct} genome assembly data fetched")
                    with open(zip_path, "wb") as f:
                        f.write(NCBI_response.content)
                else:
                    print(f"Error: {NCBI_response.status_code}")
            else:
                print(
                    f"Success: {GCF_id_correct} genome assembly data zip already exists"
                )
            err = os.system(
                f"unzip -o {zip_path} -d {os.path.join(NCBI_dir, GCF_id_correct)}"
            )
            if err:
                print(
                    f"Error: {GCF_id_correct} genome assembly data extraction failed, retrying"
                )
                os.remove(zip_path)
            version -= 1
        else:
            print(f"Success: {GCF_id} genome assembly data already exists")
            break
    else:
        src_path = os.path.join(
            NCBI_dir,
            GCF_id_correct,
            "ncbi_dataset",
            "data",
            GCF_id_correct,
            "genomic.gbff",
        )
        shutil.copy(
            src_path,
            os.path.join(
                NCBI_dir,
                GCF_id_correct + ".gbff",
            ),
        )
        shutil.rmtree(
            os.path.join(
                NCBI_dir,
                GCF_id_correct,
            )
        )
        print(f"Success: {GCF_id_correct} genome assembly data extracted")
        os.remove(zip_path)
    if GCF_id != GCF_id_correct:
        proteome_df.loc[
            proteome_df["Genome assembly ID"] == GCF_id, "Genome assembly ID"
        ] = GCF_id_correct
        print(f"Info: {GCF_id} genome assembly ID corrected to {GCF_id_correct}")
proteome_df.to_csv(os.path.join(preprocess_data_dir, "proteome_df.csv"), index=False)

In [ ]:
# Check genome assembly data
NCBI_protein_df = pd.DataFrame(
    columns=[
        "protein_id",
        "taxon",
        "GCF_id",
        "seq",
        "cds",
    ]
)
for GCF_id in proteome_df["Genome assembly ID"]:
    print(f"Info: Checking {GCF_id}")
    gbff_path = os.path.join(NCBI_dir, GCF_id + ".gbff")
    if os.path.exists(gbff_path):
        gbff = parse(gbff_path, "genbank")
        count = 0
        for record in gbff:
            for feature in record.features:
                if feature.type == "source":
                    taxon = feature.qualifiers["db_xref"][0].split(":")[-1]
                if feature.type == "CDS" and feature.qualifiers.get("pseudo") is None:
                    cds = str(feature.extract(record.seq))
                    if len(cds) != len(feature.qualifiers["translation"][0]) * 3 + 3:
                        print(feature.qualifiers["protein_id"][0])
                        print(feature.qualifiers["translation"][0])
                        print(cds.translate())
                    else:
                        NCBI_protein_df.loc[len(NCBI_protein_df)] = {
                            "protein_id": feature.qualifiers["protein_id"][0],
                            "taxon": taxon,
                            "GCF_id": GCF_id,
                            "seq": feature.qualifiers["translation"][0],
                            "cds": cds,
                        }
                        count += 1
        print(f"Success: {count} proteins extracted from {GCF_id}")
    else:
        print(f"Error: {gbff_path} not found")
        gbff.close()

# NCBI_protein_df["id"] = NCBI_protein_df.index
# NCBI_protein_df = NCBI_protein_df[
#     ["id", "protein_id", "taxon", "GCF_id", "seq", "cds"]
# ]
NCBI_protein_df.to_csv(
    os.path.join(preprocess_data_dir, "NCBI_protein_df.csv"), index=False
)
NCBI_protein_df.head()

In [ ]:
# Uniprot protein URL
Uniprot_protein_template = "https://rest.uniprot.org/uniprotkb/stream?fields=accession%2Corganism_id%2Clength%2Cxref_refseq%2Csequence&format=tsv&query=%28%28proteome%3A{}%29%29"

In [ ]:
# Fetch Uniprot protein info
Uniprot_protein_df = pd.DataFrame()
for id, row in proteome_df.iterrows():
    UP = row["Proteome Id"]
    Uniprot_protein_id_map_query_URL = Uniprot_protein_template.format(UP)
    Uniprot_response = requests.get(Uniprot_protein_id_map_query_URL)
    if Uniprot_response.status_code == 200:
        part_df = pd.read_csv(StringIO(Uniprot_response.text), sep="\t")
        part_df["GCF_id"] = row["Genome assembly ID"]
        Uniprot_protein_df = pd.concat([Uniprot_protein_df, part_df])
        print(f"Success: {UP} protein id map entries fetched")
    else:
        print(f"Error: {Uniprot_response.status_code}")
Uniprot_protein_df = Uniprot_protein_df[
    ["Entry", "Organism (ID)", "GCF_id", "Length", "RefSeq", "Sequence"]
]
# Uniprot_protein_df = Uniprot_protein_df.reset_index()
# Uniprot_protein_df["id"] = Uniprot_protein_df.index
# Uniprot_protein_df = Uniprot_protein_df[
#     ["id", "Entry", "Organism (ID)", "GCF_id", "Length", "RefSeq", "Sequence"]
# ]
Uniprot_protein_df.to_csv(
    os.path.join(preprocess_data_dir, "Uniprot_protein_df.csv"), index=False
)
Uniprot_protein_df.head()

In [ ]:
# AFDB query API
AFDB_api_template = "https://alphafold.ebi.ac.uk/api/prediction/{}"

In [ ]:
# Download AFDB structures
AFDB_dir = os.path.join(data_dir, "AFDB")
if not os.path.exists(AFDB_dir):
    os.makedirs(AFDB_dir)
AFDB_protein_df = pd.DataFrame(columns=["Entry", "model", "seq"])
errors = []
for _, row in tqdm(Uniprot_protein_df.iterrows(), total=len(Uniprot_protein_df)):
    AFDB_api = AFDB_api_template.format(row["Entry"])
    AFDB_response = requests.get(AFDB_api)
    if AFDB_response.status_code == 200:
        AFDB_response_json = AFDB_response.json()
        for d in AFDB_response_json:
            cif_url = d["cifUrl"]
            model = os.path.split(os.path.splitext(cif_url)[0])[1]
            seq = d["sequence"]
            AFDB_protein_df.loc[len(AFDB_protein_df)] = [row["Entry"], model, seq]
            if not os.path.exists(os.path.join(AFDB_dir, model + ".cif")):
                with open(os.path.join(AFDB_dir, model + ".cif"), "wb") as f:
                    f.write(requests.get(cif_url).content)
    else:
        errors.append(row["Entry"])
        print(f"Error: {row['Entry']} {AFDB_response.status_code}")
print(errors)
AFDB_protein_df.to_csv(
    os.path.join(preprocess_data_dir, "AFDB_protein_df.csv"), index=False
)

In [ ]:
AFDB_protein_df = pd.read_csv(os.path.join(preprocess_data_dir, "AFDB_protein_df.csv"))

In [ ]:
ver = AFDB_protein_df["model"].str.split("_").str[-1]
ver.groupby(ver).size()

In [ ]:
vmax = "v6"
AFDB_protein_df_vmax = AFDB_protein_df[
    AFDB_protein_df["model"].str.split("_").str[-1] == vmax
]
AFDB_protein_df_vmax.to_csv(
    os.path.join(preprocess_data_dir, "AFDB_protein_df_vmax.csv"), index=False
)

In [ ]:
# Extract same seqs from NCBI and AFDB
NCBI_Seqs = NCBI_protein_df["seq"].drop_duplicates()
AFDB_Seqs = AFDB_protein_df_vmax["seq"].drop_duplicates()
print(f"NCBI: {len(NCBI_Seqs)} AFDB: {len(AFDB_Seqs)}")
Seqs = pd.merge(NCBI_Seqs, AFDB_Seqs, how="inner", left_on="seq", right_on="seq")
print(f"Same seqs: {len(Seqs)}")
Seqs["length"] = Seqs["seq"].apply(len)
Seqs = Seqs[Seqs["length"] >= 50]
print(f"Seqs Length >= 50: {len(Seqs)}")
Seqs.sort_values(by=["length"], inplace=True)
Seqs = Seqs[["length", "seq"]].reset_index(drop=True)
Seqs["id"] = Seqs.index
Seqs = Seqs[["id", "length", "seq"]]
Seqs.head()

In [ ]:
NCBI_clean_df = NCBI_protein_df[NCBI_protein_df["seq"].isin(Seqs["seq"])]
AFDB_clean_df = AFDB_protein_df[AFDB_protein_df["seq"].isin(Seqs["seq"])]
print(f"NCBI: From {len(NCBI_protein_df)} cleaned to {len(NCBI_clean_df)}")
print(f"AFDB: From {len(AFDB_protein_df)} cleaned to {len(AFDB_clean_df)}")
NCBI_clean_df = NCBI_clean_df.reset_index()
NCBI_clean_df["id"] = NCBI_clean_df.index
NCBI_clean_df = NCBI_clean_df[
    ["id", "protein_id", "taxon", "GCF_id", "seq", "cds"]
]
AFDB_clean_df = AFDB_clean_df.reset_index()
AFDB_clean_df["id"] = AFDB_clean_df.index
AFDB_clean_df = AFDB_clean_df[
    ["id", "Entry", "model", "seq"]
]
NCBI_clean_df.to_csv(
    os.path.join(preprocess_data_dir, "NCBI_clean_df.csv"), index=False
)
AFDB_clean_df.to_csv(
    os.path.join(preprocess_data_dir, "AFDB_clean_df.csv"), index=False
)

In [ ]:
# Get NCBI and AFDB protein ids for same seqs
NCBI_seq_index_map = NCBI_clean_df.groupby("seq")["id"].apply(list).to_dict()
Seqs["NCBI_protein_ids"] = Seqs["seq"].map(NCBI_seq_index_map)
AFDB_seq_index_map = AFDB_clean_df.groupby("seq")["id"].apply(list).to_dict()
Seqs["AFDB_protein_ids"] = Seqs["seq"].map(AFDB_seq_index_map)
Seqs.to_csv(os.path.join(preprocess_data_dir, "all_protein_df.csv"), index=False)
Seqs.head()

In [ ]:
# Check count
NCBI_id_cnt = 0
AFDB_id_cnt = 0
for id, row in Seqs.iterrows():
    NCBI_protein_ids = row["NCBI_protein_ids"]
    AFDB_protein_ids = row["AFDB_protein_ids"]
    NCBI_id_cnt += len(NCBI_protein_ids)
    AFDB_id_cnt += len(AFDB_protein_ids)
print(f"NCBI: {NCBI_id_cnt} AFDB: {AFDB_id_cnt}")

In [ ]:
NCBI_clean_df.to_pickle(os.path.join(preprocess_data_dir, "NCBI_clean_df.pkl"))
AFDB_clean_df.to_pickle(os.path.join(preprocess_data_dir, "AFDB_clean_df.pkl"))
Seqs.to_pickle(os.path.join(preprocess_data_dir, "all_protein_df.pkl"))

# PART2
Generating Codon table and k value using reference species

In [ ]:
import os
from typing import DefaultDict
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("..")
from srcs.utils.utils import k_calculator

In [ ]:
data_dir = os.path.join("..", "data")
preprocess_data_dir = os.path.join(data_dir, "preprocess")
info_dir = os.path.join(data_dir, "info")
reference_entry = "K12"

In [ ]:
proteome_df = pd.read_csv(os.path.join(preprocess_data_dir, "proteome_df.csv"))
NCBI_protein_df = pd.read_csv(os.path.join(preprocess_data_dir, "NCBI_protein_df.csv"))
proteome_entries = {
    "K12": "UP000000625",
    "Sakai": "UP000000558",
    "EAEC": "UP000000746",
    "APEC": "UP000008216",
    "ETEC": "UP000001122",
    "EPEC": "UP000008205",
    "AIEC": "UP000008614",
    "ExPEC": "UP000000747",
    "UPEC": "UP000001410",
    # "HS": "UP000001123",
}

In [ ]:
# Make Codon table using reference species
reference_genome_id = proteome_df[
    proteome_df["Proteome Id"] == proteome_entries[reference_entry]
]["Genome assembly ID"].values[0]
reference_genome_id

In [ ]:
# Count codons from reference genome
codon_table = DefaultDict(int)
reference_genome = NCBI_protein_df[NCBI_protein_df["GCF_id"] == reference_genome_id]
for _, row in reference_genome.iterrows():
    cds = row["cds"]
    for i in range(0, len(cds), 3):
        codon = cds[i : i + 3]
        codon_table[codon] += 1
codon_table = pd.DataFrame.from_dict(codon_table, orient="index", columns=["count"])
codon_table = codon_table.reset_index(names=["codon"])
codon_table["usage"] = codon_table["count"] / codon_table["count"].sum()
codon_table["r_codon"] = codon_table["codon"].str.replace("T", "U")
canonical_bases = ["A", "C", "G", "U"]
canonical_codons = [
    "".join([a, b, c])
    for a in canonical_bases
    for b in canonical_bases
    for c in canonical_bases
]
codon_table = codon_table[codon_table["r_codon"].isin(canonical_codons)]
codon_table = codon_table.sort_values(by=["codon"])
codon_table

In [ ]:
# Load tRNA table
tRNA_table = pd.read_csv(os.path.join(info_dir, "tRNA.csv"))
tRNA_codon_table = (
    tRNA_table[tRNA_table["id"] > 0]
    .assign(codon=tRNA_table["codon"].str.split("/"))
    .explode("codon")[["trna", "codon"]]
)
print(tRNA_table)
print(tRNA_codon_table)

In [ ]:
tRNA_table["concentration"] = (
    tRNA_table["concentration"]
    / tRNA_table.loc[tRNA_table["trna"] == "Metf", "concentration"].values[0]
)

codon_table["k"] = codon_table["r_codon"].apply(lambda x: k_calculator(x, tRNA_table, tRNA_codon_table, codon_table))
avail_codon_mask = codon_table["r_codon"].isin(tRNA_codon_table["codon"].values)
codon_table.loc[~avail_codon_mask, "k"] = pd.NA
avail_codon_mask = ~codon_table["k"].isna()

# Regression
regression_df = codon_table.dropna()
X = regression_df["usage"].values * 100
X = X.reshape(-1, 1)
y = regression_df["k"].values
reg = LinearRegression().fit(X, y)
X_line = np.linspace(0, 6).reshape(-1, 1)
y_line = reg.predict(X_line)
X_reg = codon_table.loc[~avail_codon_mask, "usage"].values.reshape(-1, 1) * 100
y_reg = reg.predict(X_reg)
codon_table.loc[~avail_codon_mask, "k"] = y_reg
codon_table["Regression"] = pd.NA
codon_table.loc[~avail_codon_mask, "Regression"] = True

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X, y, color="blue", label="codons", s=3, alpha=0.5)
ax.plot(X_line, y_line, color="coral", label="regression", linewidth=1)
ax.scatter(X_reg, y_reg, color="red", label="codons_regression", s=3, alpha=0.5)
ax.set_xlabel("usage (%)")
ax.set_ylabel("k")
ax.set_xlim(0, 6)
ax.legend()
ax.grid()
ax.set_title(
    f"k = {reg.coef_[0]:.5f} * usage (%) + {reg.intercept_:.5f}, R^2 = {reg.score(X, y):.5f}"
)
fig.tight_layout()
fig.savefig(os.path.join(info_dir, "codon_k_regression.png"))
codon_table.to_csv(os.path.join(info_dir, "codon.csv"), index=False)

# PART3
Processing NCBI and AFDB data to pkl

In [ ]:
import numpy as np
import pickle
from Bio.Seq import Seq
import os
import pandas as pd
from Bio.PDB import MMCIFParser
from tqdm import tqdm
from Bio.Data.IUPACData import protein_letters_3to1, protein_letters_3to1_extended
import warnings
import sys
sys.path.append("..")
from srcs.utils.utils import tempo_calculator, codon_list, aa_alphabet

warnings.filterwarnings("ignore", message="'where' used without 'out'")

In [ ]:
data_dir = os.path.join("..", "data")
preprocess_data_dir = os.path.join(data_dir, "preprocess")
info_dir = os.path.join(data_dir, "info")
NCBI_pkl_dir = os.path.join(data_dir, "NCBI_pkl")
AFDB_pkl_dir = os.path.join(data_dir, "AFDB_pkl")
if not os.path.exists(NCBI_pkl_dir):
    os.makedirs(NCBI_pkl_dir)
if not os.path.exists(AFDB_pkl_dir):
    os.makedirs(AFDB_pkl_dir)

In [ ]:
NCBI_clean_df: pd.DataFrame = pd.read_pickle(
    os.path.join(preprocess_data_dir, "NCBI_clean_df.pkl")
)

In [ ]:
# Process NCBI data to pkl
NCBI_err_list = []
for _, row in tqdm(NCBI_clean_df.iterrows(), total=len(NCBI_clean_df)):
    try:
        cds = row["cds"]
        seq = row["seq"]
        codon_array = []
        aa_array = []
        for i in range(0, len(cds), 3):
            codon = cds[i : i + 3]
            if codon not in codon_list:
                codon_array.append(-1)
            else:
                codon_array.append(codon_list.index(codon))
        for i in range(0, len(seq)):
            aa = seq[i]
            if aa not in aa_alphabet:
                aa_array.append(-1)
            else:
                aa_array.append(aa_alphabet.index(aa))
        tempo = tempo_calculator(cds)
        assert (
            len(codon_array) - 1 == len(aa_array) == len(tempo)
        ), f"{row['id']}: {len(codon_array)}, {len(aa_array)}, {len(tempo)}"
        data = {
            "codon": np.array(codon_array, dtype=np.int32),
            "aa": np.array(aa_array, dtype=np.int32),
            "tempo": np.array(tempo, dtype=np.float32),
        }
        with open(os.path.join(NCBI_pkl_dir, f"NCBI_{row['id']}.pkl"), "wb") as f:
            pickle.dump(data, f)
    except Exception as e:
        print(f"Error: {row['id']}, {str(e)}")
        NCBI_err_list.append(row["id"])
        raise e
NCBI_err_list

In [ ]:
AFDB_cif_dir = os.path.join(data_dir, "AFDB")
AFDB_clean_df: pd.DataFrame = pd.read_pickle(
    os.path.join(preprocess_data_dir, "AFDB_clean_df.pkl")
)

In [ ]:
# Process AFDB data to pkl
AFDB_err_list = []
Upper_protein_letters_3to1 = {x[0].upper(): x[1] for x in protein_letters_3to1.items()}
Upper_protein_letters_3to1_extended = {
    x[0].upper(): x[1] for x in protein_letters_3to1_extended.items()
}
for _, row in tqdm(AFDB_clean_df.iterrows(), total=len(AFDB_clean_df)):
    try:
        parser = MMCIFParser()
        cif = os.path.join(AFDB_cif_dir, f"{row['model']}.cif")
        with open(cif, "r") as f:
            structures = parser.get_structure("x", cif)

            _xyz = []
            chain_idx = []
            b_factors = []
            seq = ""
            aa_array = []
            for structure_idx, structure in enumerate(structures):
                structure.atom_to_internal_coordinates()
                for chain in structure.get_chains():
                    for residue in chain.get_residues():
                        if (
                            "N" in residue
                            and "C" in residue
                            and "O" in residue
                            and "CA" in residue
                        ) and residue.id[0] == " ":
                            _xyz.append(
                                [
                                    residue["N"].coord,
                                    residue["CA"].coord,
                                    residue["C"].coord,
                                    residue["O"].coord,
                                ]
                            )
                            chain_idx.append(structure_idx)
                            b_factors.append(residue["CA"].get_bfactor())
                            seq += Upper_protein_letters_3to1_extended[residue.resname]

            assert seq == row["seq"], f"{row['model']}: seq != row['seq']"

            xyz = np.array(_xyz, dtype=np.float32)
            chain_idx = np.array(chain_idx, dtype=np.int32)
            b_factors = np.array(b_factors, dtype=np.float32)
            for i in range(0, len(seq)):
                aa = seq[i]
                if aa not in aa_alphabet:
                    aa_array.append(-1)
                else:
                    aa_array.append(aa_alphabet.index(aa))
            aa_array = np.array(aa_array, dtype=np.int32)
            assert (
                len(_xyz) == len(chain_idx) == len(b_factors) == len(aa_array)
            ), f"{row['model']}: {len(_xyz)}, {len(chain_idx)}, {len(b_factors)}, {len(aa_array)}"
            data = {
                "xyz": xyz,
                "chain_idx": chain_idx,
                "b_factor": b_factors,
                "aa": aa_array,
            }
            with open(
                os.path.join(AFDB_pkl_dir, f"AFDB_{row['id']}.pkl"), "wb"
            ) as f:
                pickle.dump(data, f)
    except Exception as e:
        print(f"Error: {row['model']}, {str(e)}")
        AFDB_err_list.append(row["model"])
        raise e
AFDB_err_list